# Does soft voting beat the individual classifiers?

This notebook measures it rather than assuming it, and it is written to report
either answer. Soft voting is not a trick that cannot work: averaging genuinely
different models usually buys a little. The question is whether these five
models, on these five columns, are different enough for it to buy anything
here.

## The premise soft voting rests on

An average helps when its members make **different** mistakes. If every member
ranks the peptides almost identically, averaging reproduces that same ranking
and ROC-AUC barely moves. So the first thing measured is the rank correlation
between members, pooled and within protein. That number decides in advance how
much room an ensemble has.

## What is compared

All on the frozen combination, `PROP + LEN`, with protein-grouped five-fold
cross-validation and the same eight-candidate tuning each model receives in
phase 1.

- the five members on their own
- **soft vote**: the mean predicted probability
- **rank vote**: the mean within-protein rank, which ignores calibration.
  Probabilities from a logistic regression, a forest and a boosted ensemble are
  not on a common scale, so a plain average can be dominated by whichever
  member is most confident. Rank voting removes that objection.

## Two evaluations, because one of them is unfair

**Pooled and per-protein ROC-AUC.** The per-protein figure is the deployment
task: ranking peptides inside one protein.

**Best single, chosen honestly.** Comparing an ensemble against "the best of
five" is biased **against the ensemble**, because the best of five is a maximum
over five noisy numbers and is inflated by that alone. The nested comparison
fixes it: for each outer fold the best member is chosen using only the other
folds, then scored on the held-out one. That is what a practitioner could
actually do, and it is the comparison the ensemble deserves.

## How to read the output

If the member correlation is high, expect the ensemble to land between the best
and the mean of its members, which is not an improvement worth reporting. If it
is low and the ensemble still gains nothing, that is a stronger negative. If
the ensemble does win, the verdict prints the margin next to its spread across
proteins so the size can be judged.

## Cost

Five models times eight tuning candidates times five folds, on five columns:
about two hundred fits of cheap models. It times one fit and prints an estimate
before starting.

This notebook writes nothing into `ML/artifacts` and changes no module. It
reads B-cell only; SARS is never loaded.


In [ ]:
# ---- 1. Get the pipeline ----------------------------------------------------
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/azhar416/B-Cell-Epitopes.git"
PROJECT_DIR = Path("/kaggle/working/B-Cell-Epitopes")

if not (PROJECT_DIR / "ML" / "train.py").is_file():
    if PROJECT_DIR.exists():
        raise FileNotFoundError(
            f"{PROJECT_DIR} exists but holds no ML pipeline. Remove it, or "
            "point PROJECT_DIR at the repository root."
        )
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)])
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import itertools
import time

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score

from ML.CFG import CFG
from ML.dataset import load_labeled_dataset, make_cross_validation_splits
from ML.experiment_matrix import MODEL_TYPES, report_environment
from ML.model import build_model
from ML.pre_dataset import build_feature_frame
from ML.selection import apply_parameters, resolve_scale_pos_weight

# ---- 2. Settings ------------------------------------------------------------
FROZEN_GROUPS = ("prop", "len")   # what phase 1 froze
N_SPLITS = 5
RANDOM_SEED = 42
BIG_PROTEIN_MIN = 100             # SARS has 467 peptides; this is its size class
TUNE = True                       # the same 8-candidate rule phase 1 uses

# Feature fusion. ProtT5 contributes 1024 mean-pooled dimensions per peptide;
# FUSION is those concatenated with the five frozen columns, which is early
# fusion. Late fusion -- voting across the two spaces separately -- is built
# further down from the same out-of-fold vectors, so both forms are measured
# on identical folds. The vectors are read from the cache committed under
# ML/assets, so cloning the repository is enough and no encoder runs.
FUSE_EMBEDDING = True
EMBEDDING_ASSET_DIR = PROJECT_DIR / "ML" / "assets" / "perbandingan_5"

# Measured on the real data, one grouped fold per model per space: PROP + LEN
# costs 0.3 min per 5-fold pass over all five models, ProtT5 7.3 min, FUSION
# 7.0 min. So defaults-only over the three spaces is ~15 min and the full
# 8-candidate rule is ~2 h, which fits a session -- and it keeps the three
# spaces on equal footing, which matters more than the time. Set this to False
# for a ~15 min pass, but then PROP + LEN is tuned and the embedding spaces are
# not, the advantage lies with PROP + LEN, and section 5 says so.
TUNE_EMBEDDING = True

# Multi-view voting: hold the CLASSIFIER fixed and vary the representation,
# the opposite axis from the sections above. The headline pair is the one
# worth asking about -- PROP + LEN against PROP + LEN + PROTGLOBAL + PROTLEN
# -- and two peptide-level views are included because they cost little and a
# two-member vote is a thin ensemble. Trim VIEWS to the first two for the
# narrow question alone.
VIEW_VOTING = True
VIEW_MODEL = "xgboost"
VIEWS = {
    "PL": {"use_prop_features": True, "use_len_features": True},
    "PLP": {"use_prop_features": True, "use_len_features": True,
            "use_protglobal_features": True, "use_protlen_features": True},
    "AAC": {"use_aac_features": True},
    "FLANK": {"use_flank_features": True},
}
VIEW_PAIR = ("PL", "PLP")         # the two-member vote reported on its own

base_cfg = CFG(DATASET_NAME="bcell")
print("Environment:")
if not report_environment():
    print()
    print("NOT the pinned image. Tolerated here: every model and every ensemble")
    print("below is measured in this same environment and on the same folds, so")
    print("a version difference moves them together.")
print()


def _find_data_dir(filename):
    bundled = PROJECT_DIR / "dataset"
    if (bundled / filename).is_file():
        return bundled
    found = sorted(Path("/kaggle/input").rglob(filename))
    if len(found) != 1:
        raise FileNotFoundError(f"Expected exactly one {filename}; found {len(found)}.")
    return found[0].parent


DATA_DIR = _find_data_dir("input_bcell.csv")

# ---- 3. Data, frozen columns, folds -----------------------------------------
# Loading with feature_set="perbandingan_5" attaches the 1024 prott5_* columns
# by a one-to-one left merge on the normalized peptide, leaving every other
# column, the row order and the row count exactly as the "proposed" load gives
# them. The assertions below hold that claim rather than trusting it.
frame = load_labeled_dataset(
    "bcell", DATA_DIR, max_peptide_length=base_cfg.MAX_PEPTIDE_LENGTH,
    feature_set="perbandingan_5" if FUSE_EMBEDDING else "proposed",
    prottrans_embedding_dir=EMBEDDING_ASSET_DIR if FUSE_EMBEDDING else None,
)
features = build_feature_frame(
    frame, feature_set="proposed",
    use_prop_features="prop" in FROZEN_GROUPS,
    use_len_features="len" in FROZEN_GROUPS,
    use_aac_features=False, use_dpc_features=False, use_flank_features=False,
    use_peptide_features=False, use_protein_features=False,
    use_biological_features=False,
)
if features.shape[1] != 5:
    raise RuntimeError(
        f"Expected the five frozen columns, built {features.shape[1]}."
    )

SPACE_LABEL = {"PL": "PROP + LEN", "T5": "ProtT5", "FUS": "FUSION"}
spaces = {"PL": features}
if FUSE_EMBEDDING:
    embedding = build_feature_frame(frame, feature_set="perbandingan_5")
    spaces["T5"] = embedding
    spaces["FUS"] = pd.concat([features, embedding], axis=1)
    if spaces["FUS"].shape[1] != features.shape[1] + embedding.shape[1]:
        raise RuntimeError("Fusion changed the column count; names collided.")
    if spaces["FUS"].isna().to_numpy().any():
        raise RuntimeError("Fusion produced missing values; rows misaligned.")

labels = frame["target"].to_numpy(dtype=int)
groups = frame["parent_protein_id"].to_numpy()
folds, strategy = make_cross_validation_splits(
    labels, groups, "bcell", n_splits=N_SPLITS, random_state=RANDOM_SEED,
)
fold_of = np.empty(len(labels), dtype=int)
for number, (_, test_index) in enumerate(folds, start=1):
    fold_of[test_index] = number

index_of = {}
for position, protein in enumerate(groups):
    index_of.setdefault(protein, []).append(position)
index_of = {k: np.array(v) for k, v in index_of.items()}
usable = [p for p, i in index_of.items() if labels[i].min() != labels[i].max()]
big = [p for p in usable if len(index_of[p]) >= BIG_PROTEIN_MIN]

print(f"rows {len(frame):,}   proteins {len(index_of):,}   split {strategy}")
print(f"columns {features.shape[1]} ({', '.join(features.columns)})")
for key, matrix in spaces.items():
    print(f"  {SPACE_LABEL[key]:<12} {matrix.shape[1]:>5} columns")


def build_view(flags):
    """One feature representation, every group off unless the view asks for it."""
    base = dict(
        use_aac_features=False, use_dpc_features=False, use_prop_features=False,
        use_flank_features=False, use_len_features=False,
        use_peptide_features=False, use_protein_features=False,
        use_biological_features=False,
    )
    base.update(flags)
    return build_feature_frame(frame, feature_set="proposed", **base)


def constant_within_protein(matrix):
    """Columns that never vary inside a protein, so they rank nothing within it."""
    return [
        column for column in matrix.columns
        if pd.DataFrame({"g": groups, "v": matrix[column]})
        .groupby("g")["v"].nunique().max() == 1
    ]


view_frames = {}
if VIEW_VOTING:
    missing = [name for name in VIEW_PAIR if name not in VIEWS]
    if missing:
        raise RuntimeError(f"VIEW_PAIR names absent from VIEWS: {missing}")
    print()
    print(f"Views for {VIEW_MODEL} (one classifier, several representations):")
    for name, flags in VIEWS.items():
        view_frames[name] = build_view(flags)
        frozen_cols = constant_within_protein(view_frames[name])
        note = (f"   {len(frozen_cols)} constant within every protein"
                if frozen_cols else "")
        print(f"  {name:<7} {view_frames[name].shape[1]:>4} columns{note}")
    pair_extra = [
        column for column in view_frames[VIEW_PAIR[1]].columns
        if column not in set(view_frames[VIEW_PAIR[0]].columns)
    ]
    if pair_extra:
        print(f"  {VIEW_PAIR[1]} adds over {VIEW_PAIR[0]}: "
              f"{', '.join(pair_extra)}")
        still = constant_within_protein(view_frames[VIEW_PAIR[1]][pair_extra])
        if len(still) == len(pair_extra):
            print("  Every added column is a protein-level constant. It cannot")
            print("  rank peptides inside a protein on its own, so a gain can")
            print("  only come from pooled ROC shifting whole proteins, or from")
            print("  a tree splitting on it and then treating groups of")
            print("  proteins differently. Read the per-protein column before")
            print("  the pooled one, and remember SARS is a single protein, so")
            print("  all of these are constant across the whole external set.")
print(f"proteins carrying both classes: {len(usable)}   "
      f"with >= {BIG_PROTEIN_MIN} peptides: {len(big)}")
if not big:
    # Nothing clears the size threshold, so the SARS-sized view would be
    # empty and every paired test on it undefined. Fall back rather than
    # fail, and say so, because a silent empty subset reads as a result.
    print(f"  no protein reaches {BIG_PROTEIN_MIN} peptides; the size-matched")
    print("  view falls back to all usable proteins.")
    big = list(usable)
print()


# ---- 4. Out-of-fold probabilities, one model at a time ----------------------
def oof_for(config, matrix):
    """Protein-grouped out-of-fold probabilities on one feature matrix."""
    out = np.full(len(labels), np.nan)
    for train_index, test_index in folds:
        model = build_model(
            model_type=config.MODEL_TYPE,
            random_state=config.RANDOM_STATE,
            random_forest_trees=config.RANDOM_FOREST_TREES,
            tree_max_depth=config.TREE_MAX_DEPTH,
            tree_min_samples_leaf=config.TREE_MIN_SAMPLES_LEAF,
            tree_min_samples_split=config.TREE_MIN_SAMPLES_SPLIT,
            tree_max_features=config.TREE_MAX_FEATURES,
            class_weight=config.CLASS_WEIGHT,
            logistic_regression_c=config.LOGISTIC_REGRESSION_C,
            logistic_regression_max_iter=config.LOGISTIC_REGRESSION_MAX_ITER,
            xgboost_estimators=config.XGBOOST_ESTIMATORS,
            xgboost_max_depth=config.XGBOOST_MAX_DEPTH,
            xgboost_learning_rate=config.XGBOOST_LEARNING_RATE,
            xgboost_subsample=config.XGBOOST_SUBSAMPLE,
            xgboost_colsample_bytree=config.XGBOOST_COLSAMPLE_BYTREE,
            xgboost_scale_pos_weight=resolve_scale_pos_weight(
                config, labels[train_index]),
            mlp_hidden_layer_sizes=config.MLP_HIDDEN_LAYER_SIZES,
            mlp_alpha=config.MLP_ALPHA,
            mlp_learning_rate_init=config.MLP_LEARNING_RATE_INIT,
            mlp_max_iter=config.MLP_MAX_ITER,
            mlp_early_stopping=config.MLP_EARLY_STOPPING,
            feature_set="proposed",
        )
        model.fit(matrix.iloc[train_index], labels[train_index])
        out[test_index] = model.predict_proba(matrix.iloc[test_index])[:, 1]
    return out


def tuned_oof(model_type, matrix, tune, space_label):
    """The protocol's rule: score 8 candidates on pooled OOF, keep the best.

    Candidate zero is the CFG default, exactly as ML.selection builds it, so
    the tuner can never return something worse than the configured baseline.
    With tune=False only that candidate runs.
    """
    config = CFG(DATASET_NAME="bcell", MODEL_TYPE=model_type)
    config.validate()
    overrides = (config.HYPERPARAMETER_TUNING_CANDIDATES.get(model_type, [])
                 if tune else [])
    best_score, best_oof, best_label = -np.inf, None, "defaults"
    for number, override in enumerate([None] + list(overrides)):
        candidate = config if override is None else apply_parameters(config, override)
        scores = oof_for(candidate, matrix)
        score = roc_auc_score(labels, scores)
        if score > best_score:
            best_score, best_oof = score, scores
            best_label = "defaults" if override is None else str(override)
        print(f"    {space_label:<11} {model_type:<20} "
              f"candidate {number}/{len(overrides)}  pooled ROC {score:.4f}",
              flush=True)
    print(f"    {space_label:<11} {model_type:<20} -> {best_score:.4f}  "
          f"{best_label}", flush=True)
    return best_oof


def candidates_for(key):
    """How many hyperparameter candidates this space gets."""
    return 8 if (TUNE if key == "PL" else TUNE_EMBEDDING) else 1


# A 1024-column pass costs nothing like a 5-column one, so each space is timed
# separately instead of extrapolating from the cheap one.
print("Timing one 5-fold pass per space (logistic regression):")
budget = 0.0
for key, matrix in spaces.items():
    started = time.time()
    _ = oof_for(CFG(DATASET_NAME="bcell", MODEL_TYPE="logistic_regression"), matrix)
    one_pass = time.time() - started
    passes = len(MODEL_TYPES) * candidates_for(key)
    estimate = passes * one_pass * 3 / 60
    budget += estimate
    print(f"  {SPACE_LABEL[key]:<12} {one_pass:>7.1f}s x {passes:>2} passes "
          f"->  ~{estimate:.0f} min")
print(f"  total rough estimate ~{budget:.0f} min, trees and the MLP being "
      "slower than logistic regression")
print("  Set TUNE_EMBEDDING or FUSE_EMBEDDING to False if that does not fit.")
print()

space_member_oof = {}
for key, matrix in spaces.items():
    print("=" * 78)
    print(f"Members on {SPACE_LABEL[key]} ({matrix.shape[1]} columns, "
          f"{candidates_for(key)} candidate(s) each)")
    print("=" * 78)
    space_member_oof[key] = {
        model_type: tuned_oof(model_type, matrix,
                              TUNE if key == "PL" else TUNE_EMBEDDING,
                              SPACE_LABEL[key])
        for model_type in MODEL_TYPES
    }
    print()

member_oof = space_member_oof["PL"]

view_oof = {}
if VIEW_VOTING:
    print("=" * 78)
    print(f"{VIEW_MODEL} on each view ({candidates_for('PL')} candidate(s) each)")
    print("=" * 78)
    for name, matrix in view_frames.items():
        view_oof[name] = tuned_oof(VIEW_MODEL, matrix, TUNE, f"view {name}")
    print()

SHORT = {"random_forest": "RF", "logistic_regression": "LR",
         "extra_trees": "ET", "xgboost": "XGB", "mlp": "MLP"}
members = list(MODEL_TYPES)


# ---- 5. The ensembles -------------------------------------------------------
def within_protein_ranks(scores):
    """Rank inside each protein, so calibration differences cannot dominate."""
    out = np.zeros(len(scores))
    for protein, index in index_of.items():
        out[index] = stats.rankdata(scores[index]) / len(index)
    return out


space_soft, space_rank = {}, {}
for key in spaces:
    space_soft[key] = np.mean(
        [space_member_oof[key][m] for m in members], axis=0)
    space_rank[key] = np.mean(
        [within_protein_ranks(space_member_oof[key][m]) for m in members], axis=0)

soft_vote = space_soft["PL"]
rank_vote = space_rank["PL"]
candidates = {**{SHORT[m]: member_oof[m] for m in members},
              "SOFT VOTE": soft_vote, "RANK VOTE": rank_vote}

if FUSE_EMBEDDING:
    for key in ("T5", "FUS"):
        for m in members:
            candidates[f"{key} {SHORT[m]}"] = space_member_oof[key][m]
        candidates[f"{key} SOFT"] = space_soft[key]
        candidates[f"{key} RANK"] = space_rank[key]
    # Late fusion: the two spaces stay separate and their votes are combined.
    # The rank form is there because a 1024-column model and a 5-column model
    # need not produce probabilities on the same scale, and a plain mean would
    # then be decided by whichever is more confident rather than more right.
    candidates["LATE SOFT"] = np.mean([space_soft["PL"], space_soft["T5"]], axis=0)
    candidates["LATE RANK"] = np.mean(
        [within_protein_ranks(space_soft["PL"]),
         within_protein_ranks(space_soft["T5"])], axis=0)

if VIEW_VOTING:
    view_names = list(view_frames)
    for name in view_names:
        candidates[f"V:{name}"] = view_oof[name]
    candidates["VIEW SOFT"] = np.mean(
        [view_oof[name] for name in view_names], axis=0)
    candidates["VIEW RANK"] = np.mean(
        [within_protein_ranks(view_oof[name]) for name in view_names], axis=0)
    # The narrow question: just the two views, voting with each other.
    candidates["PAIR SOFT"] = np.mean(
        [view_oof[name] for name in VIEW_PAIR], axis=0)
    candidates["PAIR RANK"] = np.mean(
        [within_protein_ranks(view_oof[name]) for name in VIEW_PAIR], axis=0)


def per_protein(scores, subset):
    return np.array([roc_auc_score(labels[index_of[p]], scores[index_of[p]])
                     for p in subset])


# ---- 6. How much do the members disagree? -----------------------------------
print("=" * 78)
print("1. Member agreement -- an average can only help if they differ")
print("=" * 78)
pairs = []
for a, b in itertools.combinations(members, 2):
    pooled = stats.spearmanr(member_oof[a], member_oof[b]).statistic
    within = np.nanmean([
        stats.spearmanr(member_oof[a][index_of[p]],
                        member_oof[b][index_of[p]]).statistic
        for p in usable
    ])
    pairs.append((SHORT[a], SHORT[b], pooled, within))
print(f"{'pair':<14}{'pooled rho':>12}{'within protein':>16}")
print("-" * 44)
for a, b, pooled, within in sorted(pairs, key=lambda r: -r[3]):
    print(f"{a + ' vs ' + b:<14}{pooled:>12.3f}{within:>16.3f}")
mean_within = float(np.mean([p[3] for p in pairs]))
print(f"{'mean':<14}{np.mean([p[2] for p in pairs]):>12.3f}{mean_within:>16.3f}")
print()
if mean_within > 0.85:
    print(f"  {mean_within:.2f} within protein: the members rank peptides almost")
    print("  identically, so an average has little left to correct.")
else:
    print(f"  {mean_within:.2f} within protein: there is real disagreement here,")
    print("  so an ensemble has room to gain. Whether it does is measured below.")
print()

# ---- 7. Pooled and per-protein scores ---------------------------------------
print("=" * 78)
print("2. Every member and every ensemble, same folds")
print("=" * 78)
table = []
for name, scores in candidates.items():
    table.append({
        "model": name,
        "pooled": roc_auc_score(labels, scores),
        f"per_protein_{len(usable)}": per_protein(scores, usable).mean(),
        f"per_protein_{len(big)}": per_protein(scores, big).mean(),
    })
scores_table = pd.DataFrame(table).set_index("model")
print(scores_table.round(4).to_string())
print()

# ---- 7a. Per fold, reported the way phase 1 reports it ----------------------
print("=" * 78)
print("2a. The same five folds, mean +/- std as phase 1 reports it")
print("=" * 78)
print("  Each fold's score is the out-of-fold vector restricted to that fold's")
print("  held-out rows, so it is the quantity phase 1 averaged: a model, or an")
print("  average of models, scored on rows and proteins it never trained on.")
print("  std is the population std across the five folds (np.std, ddof=0) and")
print("  SE = std / sqrt(folds), matching ML/train.py and select_configuration.")
print()


def per_fold(scores):
    """ROC-AUC on each held-out fold, taken from the out-of-fold vector."""
    out = []
    for number in range(1, N_SPLITS + 1):
        mask = fold_of == number
        if len(np.unique(labels[mask])) < 2:
            out.append(np.nan)
            continue
        out.append(roc_auc_score(labels[mask], scores[mask]))
    return np.array(out, dtype=float)


def fold_summary(values):
    """mean, std and SE across folds, defined as phase 1 defines them."""
    values = np.asarray(values, dtype=float)
    count = int(np.count_nonzero(~np.isnan(values)))
    if not count:
        return float("nan"), float("nan"), float("nan")
    std = float(np.nanstd(values))
    return float(np.nanmean(values)), std, std / np.sqrt(count)


fold_rows, fold_scores = [], {}
for name, scores in candidates.items():
    values = per_fold(scores)
    fold_scores[name] = values
    usable_folds = int(np.count_nonzero(~np.isnan(values)))
    row = {"model": name}
    row.update({f"fold{i}": v for i, v in enumerate(values, start=1)})
    row["mean"] = float(np.nanmean(values))
    row["std"] = float(np.nanstd(values))
    row["se"] = row["std"] / np.sqrt(usable_folds)
    fold_rows.append(row)
fold_table = pd.DataFrame(fold_rows).set_index("model")
print(fold_table.round(4).to_string())
print()
for name in fold_table.index:
    print(f"  {name:<12} {fold_table.loc[name, 'mean']:.4f} +/- "
          f"{fold_table.loc[name, 'std']:.4f}")
print()

# Phase 1's own rule, applied to the ensemble question. The best member on the
# fold mean is a maximum over five, so it is inflated; section 4 repeats the
# comparison without that advantage.
cv_best = fold_table.loc[[SHORT[m] for m in members], "mean"].idxmax()
cv_mean = float(fold_table.loc[cv_best, "mean"])
cv_se = float(fold_table.loc[cv_best, "se"])
print(f"  best member on the fold mean : {cv_best} {cv_mean:.4f} +/- "
      f"{fold_table.loc[cv_best, 'std']:.4f}   (SE {cv_se:.4f})")
print(f"  one-SE band around it        : {cv_mean - cv_se:.4f} .. "
      f"{cv_mean + cv_se:.4f}")
for ensemble in ("SOFT VOTE", "RANK VOTE"):
    delta = float(fold_table.loc[ensemble, "mean"]) - cv_mean
    paired = fold_scores[ensemble] - fold_scores[cv_best]
    wins = int(np.nansum(paired > 0))
    placing = "ABOVE the band" if delta > cv_se else "within 1 SE"
    print(f"  {ensemble:<10} {fold_table.loc[ensemble, 'mean']:.4f}  "
          f"delta {delta:+.4f}  wins {wins}/{len(paired)} folds  -> {placing}")
print()
print("  A delta inside the one-SE band is what phase 1 treats as a tie, and")
print("  phase 1 breaks ties toward the simpler model -- which here is a single")
print("  classifier, not five of them averaged.")
print()

scores_table["cv_mean"] = fold_table["mean"]
scores_table["cv_std"] = fold_table["std"]

best_member = max(members, key=lambda m: roc_auc_score(labels, member_oof[m]))
print(f"best single member on pooled ROC: {SHORT[best_member]} "
      f"({roc_auc_score(labels, member_oof[best_member]):.4f})")
print("  -- note this is a maximum over five, so it is inflated; section 4")
print("     repeats the comparison without that advantage.")
print()

# ---- 8. Paired tests per protein --------------------------------------------
print("=" * 78)
print("3. Paired per protein: ensemble minus each member")
print("=" * 78)
results = {}
for subset, label in ((usable, f"{len(usable)} proteins"),
                      (big, f"{len(big)} proteins >= {BIG_PROTEIN_MIN} peptides")):
    print(f"\n  {label}")
    print(f"  {'comparison':<26}{'delta':>9}{'wins':>10}{'t':>8}{'p':>11}")
    print("  " + "-" * 64)
    for ensemble in ("SOFT VOTE", "RANK VOTE"):
        for m in members:
            delta = per_protein(candidates[ensemble], subset) - per_protein(
                member_oof[m], subset)
            t_stat, p_value = stats.ttest_1samp(delta, 0.0)
            mark = "  *" if p_value < 0.05 else ""
            print(f"  {ensemble + ' - ' + SHORT[m]:<26}{delta.mean():>+9.4f}"
                  f"{f'{int((delta > 0).sum())}/{len(delta)}':>10}"
                  f"{t_stat:>8.2f}{p_value:>11.2e}{mark}")
            results[(label, ensemble, SHORT[m])] = (float(delta.mean()), float(p_value))
        print()

# ---- 9. The fair comparison: best member chosen without peeking -------------
print("=" * 78)
print("4. Nested: the best member picked on other folds, not on the test fold")
print("=" * 78)
print("  This is the comparison the ensemble deserves. Picking the best of five")
print("  on the same rows it is scored on inflates the single model; here the")
print("  choice is made on the four training folds only.")
print()
rows = []
for number in range(1, N_SPLITS + 1):
    test_mask = fold_of == number
    train_mask = ~test_mask
    if len(np.unique(labels[test_mask])) < 2:
        continue
    picked = max(members,
                 key=lambda m: roc_auc_score(labels[train_mask],
                                             member_oof[m][train_mask]))
    rows.append({
        "fold": number,
        "picked": SHORT[picked],
        "best_single": roc_auc_score(labels[test_mask], member_oof[picked][test_mask]),
        "soft_vote": roc_auc_score(labels[test_mask], soft_vote[test_mask]),
        "rank_vote": roc_auc_score(labels[test_mask], rank_vote[test_mask]),
    })
nested = pd.DataFrame(rows)
nested["soft - single"] = nested.soft_vote - nested.best_single
nested["rank - single"] = nested.rank_vote - nested.best_single
print(nested.round(4).to_string(index=False))
print()
print(f"  mean soft vote minus best single : {nested['soft - single'].mean():+.4f}")
print(f"  mean rank vote minus best single : {nested['rank - single'].mean():+.4f}")
print()

# ---- 10. Verdict ------------------------------------------------------------
print("=" * 78)
print("VERDICT")
print("=" * 78)
big_label = f"{len(big)} proteins >= {BIG_PROTEIN_MIN} peptides"
soft_vs_best = results[(big_label, "SOFT VOTE", SHORT[best_member])]
rank_vs_best = results[(big_label, "RANK VOTE", SHORT[best_member])]
soft_protein = per_protein(soft_vote, big)
best_protein = per_protein(member_oof[best_member], big)
spread = float((soft_protein - best_protein).std(ddof=1))

print(f"member agreement within protein      {mean_within:.3f}")
print(f"soft vote minus best member, {len(big)} prot  {soft_vs_best[0]:+.4f}  "
      f"(p={soft_vs_best[1]:.2e}, sd across proteins {spread:.4f})")
print(f"rank vote minus best member, {len(big)} prot  {rank_vs_best[0]:+.4f}  "
      f"(p={rank_vs_best[1]:.2e})")
print(f"soft vote minus nested best single    {nested['soft - single'].mean():+.4f}")
print()
gains = [soft_vs_best[0], rank_vs_best[0], float(nested["soft - single"].mean())]
if max(gains) <= 0:
    print("No ensemble beats the single models on any of the three views.")
    print("Soft voting does not improve the final result on this data.")
elif soft_vs_best[1] >= 0.05 or abs(soft_vs_best[0]) < spread:
    print("Any gain is smaller than its spread across proteins, or not")
    print("significant. That is not a result worth reporting as an improvement.")
else:
    print("Soft voting DOES gain here, significantly and beyond its spread.")
    print("Report it, with the margin and the per-protein spread above.")
print()

if FUSE_EMBEDDING:
    print("=" * 78)
    print("5. Feature fusion: early, late, and neither")
    print("=" * 78)
    if TUNE != TUNE_EMBEDDING:
        print(f"  CAUTION: PROP + LEN ran {8 if TUNE else 1} candidate(s) per")
        print(f"  model and the embedding spaces ran {8 if TUNE_EMBEDDING else 1}.")
        print("  The advantage lies with PROP + LEN, so a fused space losing")
        print("  narrowly here is NOT settled -- set TUNE_EMBEDDING = True.")
        print()

    # Late fusion can only help to the extent the two spaces disagree.
    cross_pooled = stats.spearmanr(space_soft["PL"], space_soft["T5"]).statistic
    cross_within = np.nanmean([
        stats.spearmanr(space_soft["PL"][index_of[p]],
                        space_soft["T5"][index_of[p]]).statistic
        for p in usable
    ])
    print(f"  PROP + LEN vote vs ProtT5 vote: rho {cross_pooled:.3f} pooled, "
          f"{cross_within:.3f} within protein")
    print("  Two spaces that rank peptides alike leave late fusion nothing to")
    print("  correct; disagreement is necessary for it, not sufficient.")
    print()

    contenders = {
        f"incumbent {cv_best}": cv_best,
        "PL SOFT VOTE": "SOFT VOTE",
        "T5 SOFT": "T5 SOFT",
        "EARLY FUSION (FUS SOFT)": "FUS SOFT",
        "LATE FUSION (SOFT)": "LATE SOFT",
        "LATE FUSION (RANK)": "LATE RANK",
    }
    fusion_rows = []
    for shown, name in contenders.items():
        values = fold_scores[name]
        paired = values - fold_scores[cv_best]
        mean, std, _ = fold_summary(values)
        fusion_rows.append({
            "contender": shown,
            "mean": mean,
            "std": std,
            "delta": mean - cv_mean,
            "wins": f"{int(np.nansum(paired > 0))}/{len(paired)}",
            "verdict": ("ABOVE 1 SE" if mean - cv_mean > cv_se
                        else "within 1 SE"),
        })
    fusion_table = pd.DataFrame(fusion_rows).set_index("contender")
    print(f"  against the incumbent {cv_best} = {cv_mean:.4f} "
          f"(SE {cv_se:.4f}), on the same five folds:")
    print(fusion_table.round(4).to_string())
    print()

    ranked = [r for r in fusion_rows if r["contender"].startswith(("EARLY", "LATE"))]
    leader = max(ranked, key=lambda r: r["mean"])
    name = contenders[leader["contender"]]
    protein_delta = (per_protein(candidates[name], big)
                     - per_protein(candidates[cv_best], big))
    t_stat, p_value = stats.ttest_1samp(protein_delta, 0.0)
    print(f"  best fusion form: {leader['contender']} "
          f"{leader['mean']:.4f} +/- {leader['std']:.4f}")
    print(f"  per protein over {len(big)} proteins: {protein_delta.mean():+.4f} "
          f"(t={t_stat:.2f}, p={p_value:.2e}, "
          f"sd {protein_delta.std(ddof=1):.4f})")
    print()
    if leader["delta"] > cv_se and p_value < 0.05:
        print("  Fusion clears one SE of the incumbent AND holds up per protein.")
        print("  This is a real gain on this data. Report it, with the margin,")
        print("  the spread, and the cost: 1024 extra columns and a transformer.")
    elif leader["delta"] > cv_se:
        print("  Fusion clears one SE on the fold mean but not per protein.")
        print("  Worth pursuing, not yet reportable: check whether a few large")
        print("  proteins carry the fold mean.")
        if not TUNE_EMBEDDING:
            print("  Rerun with TUNE_EMBEDDING = True before deciding.")
    else:
        print("  No fusion form clears one standard error of the incumbent, so")
        print("  by phase 1's own rule this is a tie -- broken toward the five")
        print("  columns, which need no transformer at inference time.")
    print()
    fusion_out = Path("/kaggle/working/softvoting_fusion.csv")
    fusion_table.to_csv(fusion_out)
    print(f"  Fusion summary saved to {fusion_out}")
    print()

if VIEW_VOTING:
    print("=" * 78)
    print(f"6. Multi-view voting: one classifier ({VIEW_MODEL}), "
          "several representations")
    print("=" * 78)

    # A vote across views can only help where the views disagree.
    print("  pairwise agreement between views (Spearman, within protein):")
    for left, right in itertools.combinations(view_names, 2):
        within = np.nanmean([
            stats.spearmanr(view_oof[left][index_of[p]],
                            view_oof[right][index_of[p]]).statistic
            for p in usable
        ])
        mark = "   <- the pair asked about" if {left, right} == set(VIEW_PAIR) else ""
        print(f"    {left + ' vs ' + right:<16}{within:>8.3f}{mark}")
    print()

    best_view = max(view_names,
                    key=lambda n: fold_summary(fold_scores[f"V:{n}"])[0])
    view_rows = []
    for shown, name in (
        [(f"view {n}", f"V:{n}") for n in view_names]
        + [(f"PAIR VOTE {VIEW_PAIR[0]}+{VIEW_PAIR[1]} (soft)", "PAIR SOFT"),
           (f"PAIR VOTE {VIEW_PAIR[0]}+{VIEW_PAIR[1]} (rank)", "PAIR RANK"),
           (f"ALL {len(view_names)} VIEWS (soft)", "VIEW SOFT"),
           (f"ALL {len(view_names)} VIEWS (rank)", "VIEW RANK")]
    ):
        mean, std, _ = fold_summary(fold_scores[name])
        paired = fold_scores[name] - fold_scores[f"V:{best_view}"]
        view_rows.append({
            "contender": shown,
            "mean": mean,
            "std": std,
            "per_protein": per_protein(candidates[name], big).mean(),
            "delta": mean - fold_summary(fold_scores[f"V:{best_view}"])[0],
            "wins": f"{int(np.nansum(paired > 0))}/{len(paired)}",
        })
    view_table = pd.DataFrame(view_rows).set_index("contender")
    base_mean, base_std, base_se = fold_summary(fold_scores[f"V:{best_view}"])
    print(f"  baseline = the best single view for {VIEW_MODEL}: {best_view} "
          f"{base_mean:.4f} +/- {base_std:.4f} (SE {base_se:.4f})")
    print("  (a maximum over the views, so it is already inflated -- that")
    print("   handicaps the votes below, not the baseline)")
    print(view_table.round(4).to_string())
    print()

    votes = ["PAIR SOFT", "PAIR RANK", "VIEW SOFT", "VIEW RANK"]
    leader = max(votes, key=lambda n: fold_summary(fold_scores[n])[0])
    lead_mean = fold_summary(fold_scores[leader])[0]
    protein_delta = (per_protein(candidates[leader], big)
                     - per_protein(candidates[f"V:{best_view}"], big))
    t_stat, p_value = stats.ttest_1samp(protein_delta, 0.0)
    print(f"  best vote: {leader} {lead_mean:.4f}  "
          f"delta {lead_mean - base_mean:+.4f} vs view {best_view}")
    print(f"  per protein over {len(big)} proteins: {protein_delta.mean():+.4f} "
          f"(t={t_stat:.2f}, p={p_value:.2e})")
    print()

    pair_mean = fold_summary(fold_scores["PAIR SOFT"])[0]
    plp_mean = fold_summary(fold_scores[f"V:{VIEW_PAIR[1]}"])[0]
    pl_mean = fold_summary(fold_scores[f"V:{VIEW_PAIR[0]}"])[0]
    print(f"  the narrow question, {VIEW_MODEL}:")
    print(f"    {VIEW_PAIR[0]:<6} alone            {pl_mean:.4f}")
    print(f"    {VIEW_PAIR[1]:<6} alone            {plp_mean:.4f}")
    print(f"    the two voting (soft)    {pair_mean:.4f}   "
          f"delta vs best alone {pair_mean - max(pl_mean, plp_mean):+.4f}")
    print()
    if lead_mean - base_mean > base_se and p_value < 0.05:
        print("  A view vote clears one SE of the best single view AND holds up")
        print("  per protein. Multi-view voting earns its place here; report it")
        print("  with the margin, the spread, and which views carry it.")
    elif lead_mean - base_mean > base_se:
        print("  A view vote clears one SE on the fold mean but not per protein.")
        print("  Check whether protein-level columns are moving pooled ROC by")
        print("  shifting whole proteins -- that will not survive on SARS,")
        print("  which is one protein.")
    else:
        print("  No view vote clears one standard error of the best single")
        print("  view, so by phase 1's rule this is a tie -- broken toward the")
        print("  single representation with fewer columns.")
    print()
    view_out = Path("/kaggle/working/softvoting_views.csv")
    view_table.to_csv(view_out)
    print(f"  View summary saved to {view_out}")
    print()

out = Path("/kaggle/working/softvoting_check.csv")
scores_table.to_csv(out)
print(scores_table.round(4).to_string())
print()
print(f"Saved to {out}")
